# In-context prediction (Section 4)

**Paper section.** Section 4 (Evaluating in-context prediction accuracy) with Figure 2 and the left panel of Figure 1 (b), Appendix C.2 with Figures 9 and 10 and Table 2, and Appendices G and H with Figures 17 to 28.

**Claim.** In the paper's words, "Figure 2 shows that the KL falls rapidly over the first 5,000-10,000 tokens, and then plateaus, during which we say that the model's predictions have converged", and "the converged KL sits well below that of the 0-HMM and 1-HMM, and matches that of the k-HMM at k = 4.2-11.0 for the four main-text parametrizations".

**Experiment.** The KL divergence between the HMM's next-token probabilities and the LLM's at every position of 20,000-token sequences, for 40 HMMs, 10 sequences each, and six LLMs, against the 0-HMM and 1-HMM baselines, from `scripts/run_prediction.sh`. The crossover order k is the suffix length at which the k-HMM's converged KL equals the LLM's.

**Result.** The KL converges within the first 5,000 to 10,000 tokens and sits below both baselines. The converged KL matches a k-HMM at k between 4.2 and 11.0 for the four main-text parametrizations of Qwen 3.5 9B, and the median crossover across each family's 10 parametrizations lies between 3.4 and 4.7.

**How the notebook works.** The first code cell sets the paths and loads the belief-probe results that every notebook shares. Figures 9 and 10 are computed from the HMM definitions alone. The remaining cells read `results/kl_<model>.csv`, compute the converged KL over the last 5,000 positions and the crossover k per parametrization, and write the figures to `figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, spearmanr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.ticker import MultipleLocator

# ═══════════════════════════════════════════════════════════
# CONFIG — change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

MODEL_KEYS = ['qwen35_9b', 'qwen35_4b', 'llama_31_8b', 'llama_32_3b', 'gemma_4_e4b', 'gemma_4_e2b']
MODEL_LABELS = {
    'qwen35_9b': 'Qwen 3.5 9B', 'qwen35_4b': 'Qwen 3.5 4B',
    'llama_31_8b': 'Llama 3.1 8B', 'llama_32_3b': 'Llama 3.2 3B',
    'gemma_4_e4b': 'Gemma 4 E4B', 'gemma_4_e2b': 'Gemma 4 E2B',
}

# Representative (HMM, param) for single-param plots
TARGETS = [
    ('Mess3', 'a=0.01, x=0.02'),
    # ('Arch', 'a=0.9'),
    ('Arch', 'a=0.99'),
    ('Wing', 'a=0.98, x=0.4'),
    ('Strata', 'a=0.97, t0=0.38, t1=0.54'),
]
HMM_ORDER = ['Mess3', 'Arch', 'Wing', 'Strata']
MODEL_ORDER = list(MODEL_KEYS)
HMM_COLORS = {'Mess3': 'tab:blue', 'Arch': 'tab:orange', 'Wing': 'tab:green', 'Strata': 'tab:red'}
HMMS = ['Mess3', 'Arch', 'Wing', 'Strata']

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')
tab10 = sns.color_palette('tab10')

def fmt_tick(v):
    s = f'{v:.1f}' if v == int(v) else f'{v:.2f}'
    if -1 < v < 1 and '.' in s:
        s = s.replace('0.', '.')   # 0.97 → .97  AND  -0.45 → -.45
    return s

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load_csv(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.csv')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return pd.read_csv(path)

def load_npz(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.npz')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

def harmonize_cross(df):
    if df is None: return None
    if 'source' in df.columns:
        df = df.rename(columns={'source': 'train', 'target': 'test'})
    return df

def harmonize_wing_labels(cross_df, gt_df):
    if cross_df is None or gt_df is None: return cross_df, gt_df
    for name in ['Wing']:
        c_params = set(cross_df[cross_df['hmm']==name]['train'].unique()) if name in cross_df['hmm'].values else set()
        g_params = set(gt_df[gt_df['hmm']==name]['train'].unique()) if name in gt_df['hmm'].values else set()
        if c_params and g_params and c_params != g_params:
            mask = gt_df['hmm'] == name
            for col in ['train', 'test']:
                gt_df.loc[mask, col] = (gt_df.loc[mask, col]
                    .str.replace('x=', 'ALPHA=').str.replace('y=', 'x=').str.replace('ALPHA=', 'a='))
    return cross_df, gt_df

# ═══════════════════════════════════════════════════════════
# Auto-load all R² files
# ═══════════════════════════════════════════════════════════
r2_files = {}
for mk in MODEL_KEYS:
    df = load_csv('r2', mk)
    if df is not None:
        r2_files[mk] = df
        print(f'{MODEL_LABELS[mk]}: {len(df)} rows, HMMs={sorted(df["hmm"].unique())}')
print(f'\n{len(r2_files)} models loaded')

## Figure 9: belief-state geometries of the four families

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys
from sklearn.decomposition import PCA
sys.path.insert(0, '..')                                  # repo root, relative to plots/
from configs.hmm_configs import HMMS
from src.hmm import stationary_distribution, sample_hmm_sequence, full_bayesian_beliefs

N_SEQ, SEQ_LEN = 50, 10000

TARGETS_GEO = [
    ('Mess3',  'a=0.01, x=0.02'),
    ('Arch',   'a=0.99'),
    ('Wing',   'a=0.98, x=0.4'),
    ('Strata', 'a=0.97, t0=0.38, t1=0.54'),
]
TITLES = {
    'Mess3':  'Mess3\n$x=0.02,\\ \\alpha=0.01$',
    'Arch':   'Arch\n$\\alpha=0.99$',
    'Wing':   'Wing\n$\\alpha=0.98,\\ x=0.4$',
    'Strata': 'Strata\n$\\alpha=0.97,\\ t_0=0.38,\\ t_1=0.54$',
}

# reference frame for 4-state families: PCA fit on the seed-0 late-window true beliefs, as in Figure 3 and the grids
gd_ref = np.load(os.path.join(RESULTS_DIR, 'geom_qwen35_9b.npz'), allow_pickle=True)

def _simplex2d(B):                       # 3-state barycentric -> equilateral triangle
    v = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
    return B[:, :3] @ v

def _pca2d(B, hmm, param_str):           # 4-state (Arch) -> the paper's PCA frame
    pca = PCA(n_components=2).fit(gd_ref[f'{hmm}__{param_str}_true'])
    return pca.transform(B)

def _collect_beliefs(hmm, param_str):
    cfg = HMMS[hmm]
    param = next(p for p in cfg['params'] if cfg['label_fn'](p) == param_str)
    T = cfg['fn'](*param); T_stack = np.stack(T); pi = stationary_distribution(T)
    return np.concatenate([full_bayesian_beliefs(sample_hmm_sequence(T, pi, SEQ_LEN, seed=s).astype(np.int64),
                                                 T_stack, pi)
                           for s in range(N_SEQ)], axis=0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (hmm, param) in zip(axes, TARGETS_GEO):
    B = _collect_beliefs(hmm, param)
    colors = np.clip(B[:, :3], 0, 1)                      # first 3 belief coords as RGB
    XY = _pca2d(B, hmm, param) if B.shape[1] >= 4 else _simplex2d(B)
    ax.scatter(XY[:, 0], XY[:, 1], s=1, c=colors, linewidths=0, rasterized=True)
    ax.set_title(TITLES[hmm], fontsize=13)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/belief_geometry.pdf', bbox_inches='tight', dpi=200)
plt.show(); plt.close()

## Figure 10 and Table 2: the sweep in the entropy–mixing plane

In [ ]:
# ══ parameter sweep in the entropy–mixing plane (new notation: T, M, H̃_T, H̃_M, H̃_TM) ══
#   λ2 = second-largest |eigenvalue| of T = Σ_x T^(x);  π = stationary distribution;
#   H_T = −Σ_ij π_i T_ij log T_ij ;  H_M = −Σ_ik π_i M_ik log M_ik ;  M_ik = Σ_j T^(x_k)_ij.
#   Normalized: H̃_T = H_T/log m, H̃_M = H_M/log n, H̃_TM = (H_T+H_M)/(log m + log n).
#   The HMM definitions are reloaded so that edits to them are picked up without restarting the kernel.
sys.path.insert(0, '..')
import importlib
import src.hmm.definitions
importlib.reload(src.hmm.definitions)
import configs.hmm_configs
importlib.reload(configs.hmm_configs)
from configs.hmm_configs import HMMS
from src.hmm import stationary_distribution

COLS  = {'Mess3': 'tab:blue', 'Arch': 'tab:orange', 'Wing': 'tab:green', 'Strata': 'tab:red'}
MARKS = {'Mess3': 'o', 'Arch': 's', 'Wing': '^', 'Strata': 'D'}

pts = {f: [] for f in HMM_ORDER}
for fam in HMM_ORDER:
    cfg = HMMS[fam]
    for param in cfg['params']:
        Ts = cfg['fn'](*param)
        T = np.sum(np.stack(Ts), axis=0)                     # transition matrix over hidden states
        m = T.shape[0]; n = len(Ts)
        lam2 = np.sort(np.abs(np.linalg.eigvals(T)))[::-1][1]
        pi = stationary_distribution(Ts)
        M = np.stack([Tx.sum(axis=1) for Tx in Ts], axis=1)  # M_ik = Σ_j T^(x_k)_ij
        HT = -np.nansum(pi[:, None] * T * np.where(T > 0, np.log(T), 0.0))
        HM = -np.nansum(pi[:, None] * M * np.where(M > 0, np.log(M), 0.0))
        pts[fam].append(((HT + HM) / (np.log(m) + np.log(n)), HT / np.log(m), HM / np.log(n), lam2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
xlabels = [r'$\tilde H_{T\!M}$', r'$\tilde H_T$', r'$\tilde H_M$']
for i, ax in enumerate(axes):
    for fam in HMM_ORDER:
        arr = np.array(pts[fam])
        ax.scatter(arr[:, i], arr[:, 3], color=COLS[fam], marker=MARKS[fam],
                   s=55, alpha=0.85, edgecolor='k', linewidth=0.4, label=fam if i == 0 else None)
    for g in [0.5, 0.75, 0.95, 0.99]:
        ax.axhline(g, color='0.5', lw=1, ls='--', zorder=0)
    ax.set_xlabel(xlabels[i], fontsize=18)
    ax.tick_params(labelsize=13, length=3)
    ax.grid(alpha=0.2, lw=0.5)
axes[0].set_ylabel(r'Mixing rate $\lambda_2$', fontsize=16)
axes[0].set_ylim(0.4, 1.02)
axes[0].legend(fontsize=13, frameon=False, loc='lower left')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/all_lambda2_vs_entropy.pdf', bbox_inches='tight')
plt.show(); plt.close()
# ranges of the sweep, for Table 2
for fam in HMM_ORDER:
    arr = np.array(pts[fam])
    print(f"{fam:7s} lambda2 {arr[:,3].min():.3f} -> {arr[:,3].max():.3f}   H~TM {arr[:,0].min():.2f} -> {arr[:,0].max():.2f}   H~T {arr[:,1].min():.2f} -> {arr[:,1].max():.2f}   H~M {arr[:,2].min():.2f} -> {arr[:,2].max():.2f}")


## Figure 2: KL along the sequence and the crossover order k (Qwen 3.5 9B)

In [ ]:
MODEL = 'qwen35_9b'
kl_df = load_csv('kl', MODEL)
print(f'KL: {len(kl_df)} rows') if kl_df is not None else None

In [ ]:
# Compute KL summary from kl_df directly
last5k = kl_df[kl_df['position'] >= 15000]

per_seed = (last5k
    .groupby(['hmm', 'param', 'source', 'seed'])['KL']
    .mean().reset_index().rename(columns={'KL': 'KL_mean'}))

summary = (per_seed
    .groupby(['hmm', 'param', 'source'])
    .agg(KL_mean=('KL_mean', 'mean'), KL_std=('KL_mean', 'std'))
    .reset_index())

print(summary)


In [ ]:
target_set = set(TARGETS)
kl_repr = summary[(summary['source'] == 'LLM') &
                   (summary['hmm'] != 'Spiral') &
                   summary.apply(lambda r: (r['hmm'], r['param']) in target_set, axis=1)]
print(kl_repr[['hmm', 'param', 'KL_mean', 'KL_std']].to_string(index=False))


In [ ]:
# ── Compute KL vs k inline from HMM definitions ──
# No external file needed — pure HMM computation

def _stationary(T_matrices):
    T_full = sum(T_matrices); eigvals, eigvecs = np.linalg.eig(T_full.T)
    idx = np.argmin(np.abs(eigvals - 1.0)); pi = np.real(eigvecs[:, idx])
    return pi / pi.sum()

def _sample(T_matrices, pi, n, seed=0):
    rng = np.random.default_rng(seed); n_states, n_tok = len(pi), len(T_matrices)
    state = rng.choice(n_states, p=pi); tokens = []
    for _ in range(n):
        tp = np.array([T_matrices[z][state].sum() for z in range(n_tok)]); tp /= tp.sum()
        z = rng.choice(n_tok, p=tp); tokens.append(z)
        nsp = T_matrices[z][state] / T_matrices[z][state].sum()
        state = rng.choice(n_states, p=nsp)
    return np.array(tokens)

def _beliefs(tokens, T_stack, pi):
    n = len(tokens); b = pi.copy(); beliefs = np.zeros((n, len(pi)))
    for t in range(n):
        b = b @ T_stack[tokens[t]]; b /= b.sum(); beliefs[t] = b
    return beliefs

def _ntp(beliefs, T_matrices):
    M = np.stack([T.sum(axis=1) for T in T_matrices], axis=1)
    return beliefs @ M

def _kl(p, q, eps=1e-12):
    return np.sum(p * np.log((p + eps) / (q + eps)), axis=-1)

# ── HMM definitions (minimal, for KL vs k only) ──
def mess3(a, x):
    b = (1-a)/2; y = 1-2*x; return [
        np.array([[a*y,b*x,b*x],[a*x,b*y,b*x],[a*x,b*x,b*y]]),
        np.array([[b*y,a*x,b*x],[b*x,a*y,b*x],[b*x,a*x,b*y]]),
        np.array([[b*y,b*x,a*x],[b*x,b*y,a*x],[b*x,b*x,a*y]])]

def arch(a):
    b = (1-a)/3; return [
        np.array([[0.8*a,0,0,0],[0,0.2*a,0,0],[0,0,0.4*a,0],[0,0,0,0.6*a]]),
        np.array([[0,0,0,0],[0,0.4*a,0,0.4*b],[0,0,0.3*a,0],[0,0,0,0.16*a]]),
        np.array([[0.2*a,b,b,b],[b,0.4*a,b,0.6*b],[b,b,0.3*a,b],[b,b,b,0.24*a]])]

def wing(a, x):
    b = (1-a)/2; return [
        np.array([[0,b,0],[0,x*a,0.5*b],[b,0,0]]),
        np.array([[a,0,b],[b,(1-x)*a,0.5*b],[0,b,a]])]

def strata(a, t0, t1):
    b = (1-a)/2; return [
        np.array([[t0*a,0,0],[0,t1*a,0],[0,0,0]]),
        np.array([[(1-t0)*a,b,b],[b,(1-t1)*a,b],[b,b,a]])]

HMM_FNS = {
    'Mess3': lambda p: mess3(*p), 'Arch': lambda p: arch(*p),
    'Wing': lambda p: wing(*p), 'Strata': lambda p: strata(*p),
}

# Parse param strings back to tuples
def _parse_param(hmm, param_str):
    parts = param_str.split(', ')
    return tuple(float(p.split('=')[1]) for p in parts)

# ── Compute KL(true || k-suffix) for k=1..20 ──
SEQ_LEN, PROBE_START, N_SEEDS, K_MAX = 20000, 15000, 10, 20
kl_k_rows = []

for hmm, param_str in TARGETS:
    param = _parse_param(hmm, param_str)
    T = HMM_FNS[hmm](param); T_stack = np.stack(T); pi = _stationary(T)

    for seed in range(N_SEEDS):
        tokens = _sample(T, pi, SEQ_LEN, seed=seed).astype(np.int64)
        true_beliefs = _beliefs(tokens, T_stack, pi)
        true_ntp = _ntp(true_beliefs, T)

        for k in range(1, K_MAX + 1):
            # k-suffix beliefs
            n_late = SEQ_LEN - PROBE_START
            k_beliefs = np.zeros((n_late, len(pi)))
            for i in range(n_late):
                t = PROBE_START + i; s = max(0, t - k + 1)
                b = pi.copy()
                for j in range(s, t + 1):
                    b = b @ T_stack[tokens[j]]; b /= b.sum()
                k_beliefs[i] = b
            k_ntp = _ntp(k_beliefs, T)
            kl_vals = _kl(true_ntp[PROBE_START:], k_ntp)
            kl_k_rows.append({'hmm': hmm, 'param': param_str, 'k': k,
                              'seed': seed, 'KL': kl_vals.mean()})

kl_k = pd.DataFrame(kl_k_rows).groupby(['hmm', 'param', 'k']).agg(
    mean=('KL', 'mean'), std=('KL', 'std')).reset_index()

# ── Crossover k ──
kl_llm = summary[(summary['source'] == 'LLM') & (summary['hmm'] != 'Spiral')]
crossover_rows = []
for _, row in kl_llm.iterrows():
    sub = kl_k[(kl_k['hmm'] == row['hmm']) & (kl_k['param'] == row['param'])].sort_values('k')
    if len(sub) == 0: continue
    match_k = sub[sub['mean'] <= row['KL_mean']]['k']
    k_val = int(match_k.min()) if len(match_k) > 0 else '>20'
    crossover_rows.append({'hmm': row['hmm'], 'param': row['param'],
                           'KL_LLM': row['KL_mean'],
                           'KL_k1': sub[sub['k']==1]['mean'].values[0],
                           'crossover_k': k_val})
crossover = pd.DataFrame(crossover_rows)
print(crossover.to_string(index=False))

for hmm in HMM_ORDER:
    sub = crossover[crossover['hmm'] == hmm]
    numeric = sub[sub['crossover_k'] != '>20']['crossover_k'].astype(int)
    if len(numeric):
        print(f'  {hmm}: median k={numeric.median():.0f}, range=[{numeric.min()}, {numeric.max()}]')


In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
from matplotlib.ticker import LogLocator, LogFormatterMathtext
from matplotlib.colors import LinearSegmentedColormap

matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'], 'font.weight': 'light',
})
sns.set_context('notebook')

_base = plt.get_cmap('RdYlGn_r')
ramp = LinearSegmentedColormap.from_list(
    'gry', [_base(0.0), _base(0.25), _base(0.80), _base(1.0), _base(0.5)])
KMAX = float(kl_k['k'].max())
def kcolor(k):
    t = (k / KMAX) ** 0.4
    return ramp(min(max(t, 0.0), 1.0))
C_ORDER0 = kcolor(0)
C_ORDER1 = kcolor(1)

K_MARGIN = 2.5
def _xmax_for(hmm, param):
    c = crossover[(crossover['hmm'] == hmm) & (crossover['param'] == param)]
    if len(c) and c.iloc[0]['crossover_k'] != '>20':
        return int(np.ceil(int(c.iloc[0]['crossover_k']) + K_MARGIN))
    return int(KMAX)
def _kticks(xmax):
    for step in (5, 4, 2, 1):
        ticks = list(range(0, int(xmax) + 1, step))
        if len(ticks) >= 4:
            return ticks
    return list(range(0, int(xmax) + 1))

def _add_log_minor(ax):
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=100))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())

Y_FLOOR = 5e-3

fig = plt.figure(figsize=(8, 3.75))
outer = fig.add_gridspec(2, 2)

labels = {'LLM': 'LLM', 'Order-1': '1-HMM', 'Order-0': '0-HMM'}
tab10 = sns.color_palette('tab10')
colors = {'LLM': 'black', 'Order-1': C_ORDER1, 'Order-0': C_ORDER0}
n_seeds = 10

for i, (hmm, param) in enumerate(TARGETS):
    row, col = divmod(i, 2)
    inner = outer[row, col].subgridspec(1, 2, wspace=0.1, width_ratios=[0.6, 0.4])
    ax_pos = fig.add_subplot(inner[0])
    ax_k   = fig.add_subplot(inner[1], sharey=ax_pos)

    sub = kl_df[(kl_df['hmm'] == hmm) & (kl_df['param'] == param)]
    for source in ['Order-0', 'Order-1', 'LLM']:
        s = sub[sub['source'] == source]
        stats = s.groupby('position')['KL'].agg(['mean', 'sem']).reset_index()
        lw = 2 if source == 'LLM' else 1.5
        ax_pos.plot(stats['position'], stats['mean'], color=colors[source], lw=lw, label=labels[source])
        lo = (stats['mean'] - 1.96 * stats['sem']).clip(lower=stats['mean'].min() * 1e-2)
        hi = stats['mean'] + 1.96 * stats['sem']
        ax_pos.fill_between(stats['position'], lo, hi, color=colors[source], alpha=0.15, linewidth=0)
    ax_pos.axvspan(15000, 20000, alpha=0.2, color='gray', zorder=0)
    ax_pos.set_xticks([0, 5000, 10000, 15000, 20000])
    ax_pos.set_xticklabels(['0', '5k', '10k', '15k', '20k'])
    ax_pos.tick_params(axis='both', labelsize=11, length=3)
    ax_pos.grid(True, alpha=0.2, linewidth=0.5)
    ax_pos.set_yscale('log')
    ax_pos.set_ylim(bottom=Y_FLOOR)
    ax_pos.yaxis.set_major_locator(LogLocator(base=10, numticks=4))
    ax_pos.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
    _add_log_minor(ax_pos)
    ax_pos.tick_params(axis='y', which='minor', length=2)
    if row == 1:
        ax_pos.set_xlabel('Token Position', fontsize=11)
    if col == 0:
        ax_pos.set_ylabel('KL Divergence', fontsize=11)

    xmax = _xmax_for(hmm, param)
    sub_k = kl_k[(kl_k['hmm'] == hmm) & (kl_k['param'] == param)].sort_values('k')
    llm_kl = kl_llm[(kl_llm['hmm'] == hmm) & (kl_llm['param'] == param)]['KL_mean'].values[0]

    xk0 = sub_k['k'].values.astype(float)
    yk0 = sub_k['mean'].values
    sem_k = (sub_k['std'] / np.sqrt(n_seeds)).values
    m = xk0 <= xmax
    xk0, yk0, sem_k = xk0[m], yk0[m], sem_k[m]

    order0_kl = sub[(sub['source'] == 'Order-0') & (sub['position'] >= 15000)]['KL'].mean()
    if np.isfinite(order0_kl):
        xk = np.concatenate([[0.0], xk0]); yk = np.concatenate([[order0_kl], yk0])
    else:
        xk, yk = xk0, yk0
    _pts = np.array([xk, yk]).T.reshape(-1, 1, 2)
    _segs = np.concatenate([_pts[:-1], _pts[1:]], axis=1)
    lc = LineCollection(_segs, colors=[kcolor(k) for k in xk[:-1]], linewidths=3, zorder=3)
    ax_k.add_collection(lc)

    lo_k = (yk0 - 1.96 * sem_k).clip(min=yk0.min() * 1e-2)
    hi_k = yk0 + 1.96 * sem_k
    ax_k.fill_between(xk0, lo_k, hi_k, alpha=0.15, color='black', zorder=1)
    ax_k.axhline(llm_kl, color='black', lw=3, zorder=2)
    cross = crossover[(crossover['hmm'] == hmm) & (crossover['param'] == param)]
    if len(cross) and cross.iloc[0]['crossover_k'] != '>20':
        k_star = int(cross.iloc[0]['crossover_k'])
        kl_at = sub_k.set_index('k')['mean']
        ka, kb = kl_at[k_star - 1], kl_at[k_star]
        frac = (ka - llm_kl) / (ka - kb)
        k_cross = (k_star - 1) + frac
        if k_cross <= xmax:
            ax_k.axvline(k_cross, color='blue', lw=3, zorder=4)
            _left  = hmm in ('Arch', 'Wing', 'Strata')
            _yfrac = 0.80 if hmm in ('Wing', 'Strata') else 0.75
            ax_k.annotate(f'$k$={k_cross:.1f}', xy=(k_cross, _yfrac), xycoords=('data', 'axes fraction'),
                          fontsize=11, color='blue', ha='right' if _left else 'left',
                          xytext=(-4 if _left else 4, 0), textcoords='offset points')
    ax_k.set_xlim(-0.5, xmax + 0.5)
    ax_k.set_xticks(_kticks(xmax))
    ax_k.yaxis.set_major_locator(LogLocator(base=10, numticks=4))
    _add_log_minor(ax_k)
    ax_k.tick_params(axis='both', labelsize=11, length=3, labelleft=False)
    ax_k.tick_params(axis='y', which='minor', length=2, labelleft=False)
    ax_k.grid(True, alpha=0.2, linewidth=0.5)
    if row == 1:
        ax_k.set_xlabel('$k$ (suffix length)', fontsize=11)

    ax_pos.set_title(hmm, fontsize=12, x=1.05, pad=8)

class _GradHandle:
    def __init__(self, cmap_colors): self.cmap_colors = cmap_colors
class _GradHandler(HandlerBase):
    def create_artists(self, legend, orig, xd, yd, w, h, fontsize, trans):
        n = len(orig.cmap_colors); xs = np.linspace(xd, xd + w, n + 1); y = yd + h / 2.0
        segs = [[(xs[j], y), (xs[j + 1], y)] for j in range(n)]
        return [LineCollection(segs, colors=orig.cmap_colors, linewidths=5, transform=trans)]

_kvals = np.arange(0, int(KMAX) + 1)
_grad_handle = _GradHandle([kcolor(k) for k in _kvals])
legend_handles = [Line2D([], [], color='black', lw=5), Line2D([], [], color=C_ORDER1, lw=5),
                  Line2D([], [], color=C_ORDER0, lw=5), _grad_handle]
legend_labels = ['LLM', '1-HMM', '0-HMM', r'$k$-HMM']
fig.legend(handles=legend_handles, labels=legend_labels, loc='upper center', ncol=4,
           fontsize=11, frameon=False, bbox_to_anchor=(0.5, 0.12),      # ← raise y to pull legend toward rows
           columnspacing=1.2, handletextpad=0.5, handlelength=1.5,
           handler_map={_GradHandle: _GradHandler()})

fig.tight_layout(rect=[0, 0.08, 1, 1], h_pad=0.5)                       # ← h_pad small = rows closer; rect[1]=0.08 leaves legend strip
plt.savefig(f'{PLOT_DIR}/kl.pdf', bbox_inches='tight')
plt.savefig(f'{PLOT_DIR}/kl.png', bbox_inches='tight')
plt.show(); plt.close()

## Figure 1 (b), left panel

In [ ]:
# ══ Figure 1 panel: Wing per-position KL (standalone, no k-panel, legend below) ══
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, LogFormatterMathtext
from matplotlib.colors import LinearSegmentedColormap

HMM_F1 = 'Wing'
PARAM_F1 = [p for h, p in TARGETS if h == HMM_F1][0]
PANEL_LW = 5.0
Y_FLOOR = 5e-3

_base = plt.get_cmap('RdYlGn_r')
ramp = LinearSegmentedColormap.from_list(
    'gry', [_base(0.0), _base(0.25), _base(0.80), _base(1.0), _base(0.5)])
KMAX = float(kl_k['k'].max())
def kcolor(k):
    t = (k / KMAX) ** 0.4
    return ramp(min(max(t, 0.0), 1.0))
C_ORDER0 = kcolor(0)
C_ORDER1 = kcolor(1)

labels = {'LLM': 'LLM', 'Order-1': '1-HMM', 'Order-0': '0-HMM'}
colors = {'LLM': 'black', 'Order-1': C_ORDER1, 'Order-0': C_ORDER0}

fig, ax = plt.subplots(figsize=(8.5, 3.7))
fig.patch.set_facecolor('white')

sub = kl_df[(kl_df['hmm'] == HMM_F1) & (kl_df['param'] == PARAM_F1)]
for source in ['Order-0', 'Order-1', 'LLM']:
    s = sub[sub['source'] == source]
    stats = s.groupby('position')['KL'].agg(['mean', 'sem']).reset_index()
    lw = 6.0 if source == 'LLM' else 5.5
    ax.plot(stats['position'], stats['mean'], color=colors[source], lw=lw, label=labels[source])
    lo = (stats['mean'] - 1.96 * stats['sem']).clip(lower=stats['mean'].min() * 1e-2)
    hi = stats['mean'] + 1.96 * stats['sem']
    ax.fill_between(stats['position'], lo, hi, color=colors[source], alpha=0.15, linewidth=0)

ax.axvspan(15000, 20000, alpha=0.2, color='gray', zorder=0)
ax.set_xticks([0, 5000, 10000, 15000, 20000])
ax.set_xticklabels(['0', '5k', '10k', '15k', '20k'])
ax.set_yscale('log')
ax.set_ylim(bottom=Y_FLOOR)
ax.yaxis.set_major_locator(LogLocator(base=10, numticks=4))
ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=100))
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.grid(True, alpha=0.2, linewidth=0.5)
ax.tick_params(axis='both', labelsize=30, length=7, width=2.5)
ax.tick_params(axis='y', which='minor', length=4, width=2.0)
ax.set_xlabel('Token Position', fontsize=34)
ax.set_ylabel('KL Divergence', fontsize=31)
for spine in ax.spines.values():
    spine.set_linewidth(PANEL_LW)

legend_handles = [Line2D([], [], color=colors[s], lw=8, label=labels[s])
                  for s in ['LLM', 'Order-1', 'Order-0']]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           fontsize=30, frameon=False, bbox_to_anchor=(0.5, -0.46),
           handletextpad=0.5, handlelength=1.4, labelspacing=0.35)

plt.savefig(f'{PLOT_DIR}/fig1_kl_wing.svg', bbox_inches='tight', dpi=300)
plt.show(); plt.close()

## Appendix H: crossover k for all models (Figures 23 to 28) and the quoted ranges

In [ ]:
# ── Crossover-k histogram per family, per model: one crossover per parametrization ──
# Reuses the kl_k helpers. They're captured under collision-safe names below, because a
# plotting cell can rebind `_ntp` to a color tuple (`_ntp = tab10[4]`). Run the kl_k cell first.
import matplotlib
from matplotlib.ticker import MaxNLocator
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')

# Guard + capture: fail loudly if `_ntp` was clobbered, then alias to names no plot cell touches.
assert callable(_ntp), "`_ntp` is not a function -- re-run the kl_k cell (a later cell rebound it to a color)."
_NTP, _BELIEFS, _KL, _SAMPLE, _STAT = _ntp, _beliefs, _kl, _sample, _stationary

SEQ_LEN, PROBE_START, N_SEEDS, K_MAX = 20000, 15000, 10, 20

def _param_kcurve(hmm, param_str):
    """Seed-averaged k-HMM KL for k=1..K_MAX. Vectorized over positions (einsum)."""
    param = _parse_param(hmm, param_str)
    T = HMM_FNS[hmm](param); T_stack = np.stack(T); pi = _STAT(T)
    m = len(pi); n_late = SEQ_LEN - PROBE_START
    per_k = np.zeros((N_SEEDS, K_MAX))
    for seed in range(N_SEEDS):
        tokens = _SAMPLE(T, pi, SEQ_LEN, seed=seed).astype(np.int64)
        true_ntp = _NTP(_BELIEFS(tokens, T_stack, pi), T)[PROBE_START:]
        late = np.arange(PROBE_START, PROBE_START + n_late)
        M = T_stack[tokens]
        for ki, k in enumerate(range(1, K_MAX + 1)):
            B = np.tile(pi, (n_late, 1))
            for off in range(k):
                Msel = M[late - (k - 1) + off]
                B = np.einsum('ns,nsj->nj', B, Msel)
                B /= B.sum(1, keepdims=True)
            per_k[seed, ki] = _KL(true_ntp, _NTP(B, T)).mean()
    return np.arange(1, K_MAX + 1), per_k.mean(axis=0)

def _crossover(k_arr, kl_mean, llm_kl):
    """First k where seed-avg k-curve <= seed-avg LLM KL, linearly interpolated."""
    if not np.isfinite(llm_kl):
        return np.nan
    below = np.where(kl_mean <= llm_kl)[0]
    if len(below) == 0:
        return np.nan
    j = below[0]
    if j == 0:
        return float(k_arr[0])
    a, b = kl_mean[j - 1], kl_mean[j]
    frac = (a - llm_kl) / (a - b) if a != b else 0.0
    return float(k_arr[j - 1] + frac * (k_arr[j] - k_arr[j - 1]))

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    kdf = load_csv('kl', model_key)
    if kdf is None:
        continue
    kdf = kdf[kdf['hmm'] != 'Spiral']
    llm_pp = (kdf[(kdf['source'] == 'LLM') & (kdf['position'] >= PROBE_START)]
              .groupby(['hmm', 'param'])['KL'].mean())
    hmms = [h for h in HMM_ORDER if h in kdf['hmm'].values]

    fig, axes = plt.subplots(1, len(hmms), figsize=(16, 5), squeeze=False, sharey=True)
    axes = axes[0]
    for ai, hmm in enumerate(hmms):
        ax = axes[ai]
        params = sorted(kdf[kdf['hmm'] == hmm]['param'].unique())
        xs, n_never = [], 0
        for pp in params:
            if (hmm, pp) not in llm_pp.index:
                continue
            k_arr, kl_mean = _param_kcurve(hmm, pp)
            c = _crossover(k_arr, kl_mean, float(llm_pp.loc[(hmm, pp)]))
            if np.isfinite(c):
                xs.append(c)
            else:
                n_never += 1
        xs = np.array(xs)
        if len(xs):
            bins = np.arange(0, np.ceil(np.nanmax(xs)) + 1.5, 1.0)
            ax.hist(xs, bins=bins, color='#4575b4', edgecolor='white', linewidth=0.6)
            ax.axvline(np.median(xs), color='black', lw=3, ls='--')
            ax.axvline(np.mean(xs),  color='black', lw=3, ls='-')
            _h = [Line2D([], [], color='black', lw=3, ls='--', label=f'median = {np.median(xs):.1f}'),
                  Line2D([], [], color='black', lw=3, ls='-',  label=f'mean = {np.mean(xs):.1f}')]
            ax.legend(handles=_h, fontsize=14, loc='upper center',
                      bbox_to_anchor=(0.5, -0.28), frameon=True)
        ttl = f'{hmm}  (n={len(xs)} params'
        ttl += f', {n_never} never cross)' if n_never else ')'
        ax.set_title(ttl, fontsize=23, pad=18, y=0.95)
        ax.set_xlabel('crossover $k$', fontsize=22)
        if ai == 0:
            ax.set_ylabel('count', fontsize=22)
        ax.tick_params(axis='both', labelsize=22, length=3)
        ax.grid(True, axis='y', alpha=0.2, linewidth=0.5); ax.set_axisbelow(True)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3, integer=True))

    fig.suptitle(model_name, fontsize=22, y=1.02)
    plt.tight_layout(w_pad=0)
    plt.savefig(f'{PLOT_DIR}/crossover_hist_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
# ── Crossover summary table: per model × family -> range, median, never-crossed ──
# Run AFTER the crossover-hist cell (reuses _param_kcurve, _crossover) and cell 12.
_kcurve_cache = {}
def _kcurve_cached(hmm, pp):
    if (hmm, pp) not in _kcurve_cache:
        _kcurve_cache[(hmm, pp)] = _param_kcurve(hmm, pp)
    return _kcurve_cache[(hmm, pp)]

rows = []
for model_key in MODEL_KEYS:
    kdf = load_csv('kl', model_key)
    if kdf is None:
        continue
    kdf = kdf[kdf['hmm'] != 'Spiral']
    llm_pp = (kdf[(kdf['source'] == 'LLM') & (kdf['position'] >= PROBE_START)]
              .groupby(['hmm', 'param'])['KL'].mean())
    for hmm in HMM_ORDER:
        if hmm not in kdf['hmm'].values:
            continue
        xs, n_never = [], 0
        for pp in sorted(kdf[kdf['hmm'] == hmm]['param'].unique()):
            if (hmm, pp) not in llm_pp.index:
                continue
            k_arr, kl_mean = _kcurve_cached(hmm, pp)
            c = _crossover(k_arr, kl_mean, float(llm_pp.loc[(hmm, pp)]))
            if np.isfinite(c):
                xs.append(c)
            else:
                n_never += 1
        xs = np.array(xs)
        rows.append({
            'model': MODEL_LABELS.get(model_key, model_key),
            'family': hmm,
            'n': len(xs), 'n_never': n_never,
            'min': xs.min() if len(xs) else np.nan,
            'max': xs.max() if len(xs) else np.nan,
            'median': np.median(xs) if len(xs) else np.nan,
        })

summary = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda v: f'{v:.1f}')
print(summary.to_string(index=False))

print('\n— per model (all families pooled) —')
print(summary.groupby('model').apply(
    lambda g: pd.Series({'min': g['min'].min(), 'max': g['max'].max(),
                         'median_of_medians': g['median'].median(),
                         'n_never': g['n_never'].sum()})).to_string())

print('\n— per family (all models pooled) —')
print(summary.groupby('family').apply(
    lambda g: pd.Series({'min': g['min'].min(), 'max': g['max'].max(),
                         'median_of_medians': g['median'].median(),
                         'n_never': g['n_never'].sum()})).to_string())

In [ ]:
model_key = [k for k in MODEL_KEYS if 'e2b' in k.lower()][0]
print('using:', model_key)
kdf = load_csv('kl', model_key); kdf = kdf[kdf['hmm'] != 'Spiral']
llm_pp = (kdf[(kdf['source'] == 'LLM') & (kdf['position'] >= PROBE_START)]
          .groupby(['hmm', 'param'])['KL'].mean())
for hmm in HMM_ORDER:
    cs = []
    for pp in sorted(kdf[kdf['hmm'] == hmm]['param'].unique()):
        if (hmm, pp) not in llm_pp.index: continue
        k_arr, kl_mean = _kcurve_cached(hmm, pp)
        cs.append(_crossover(k_arr, kl_mean, float(llm_pp.loc[(hmm, pp)])))
    cs = np.array(cs)
    print(f'{hmm:8s} crossovers: {np.sort(np.round(cs,1))}   n<=1.05: {(cs<=1.05).sum()}/10')

## Appendix G: KL curves for all models and parametrizations (Figures 17 to 22)

In [ ]:
# KL divergence across all parameterizations and models

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    kl_df = load_csv('kl', model_key)
    if kl_df is None:
        continue

    data = kl_df[kl_df['hmm'] != 'Spiral']
    hmms = [h for h in HMM_ORDER if h in data['hmm'].values]

    fig = plt.figure(figsize=(13, 8))
    fig.patch.set_facecolor('white')
    fig.patch.set_edgecolor('white')
    fig.patch.set_linewidth(0)
    outer = fig.add_gridspec(2, 2, wspace=0.55, hspace=0.3)

    all_axes = []
    for j, hmm in enumerate(hmms[:4]):
        r, c = divmod(j, 2)
        ax = fig.add_subplot(outer[r, c])
        all_axes.append((ax, hmm, r, c))

        hmm_data = data[data['hmm'] == hmm]
        params = sorted(hmm_data['param'].unique())
        n_p = len(params)
        tp = np.linspace(0.3, 0.9, n_p)
        colors_llm = plt.cm.viridis(tp)
        colors_ord1 = plt.cm.Blues(tp)
        colors_ord0 = plt.cm.Reds(tp)

        for i, param in enumerate(params):
            for source, colors, lw, alpha in [
                ('Order-0', colors_ord0, 0.6, 0.5),
                ('Order-1', colors_ord1, 0.6, 0.5),
                ('LLM',     colors_llm,  1.5, 1.0),
            ]:
                sub = hmm_data[(hmm_data['param'] == param) & (hmm_data['source'] == source)]
                stats = sub.groupby('position')['KL'].agg(['mean']).reset_index()
                ax.plot(stats['position'], stats['mean'], '-', color=colors[i],
                        lw=lw, alpha=alpha)

        ax.axvspan(15000, 20000, alpha=0.08, color='gray', zorder=0)
        ax.set_title(hmm, fontsize=16)
        ax.set_xticks([0, 5000, 10000, 15000, 20000])
        ax.set_xticklabels(['0', '5k', '10k', '15k', '20k'])
        ax.tick_params(axis='both', labelsize=13, length=3)
        ax.grid(True, alpha=0.2, linewidth=0.5)

        ax.set_yscale('log')
        if r == 1: ax.set_xlabel('Token Position', fontsize=14)

        # Legend to the right
        param_handles = [Line2D([], [], color=colors_llm[i], lw=3, label=p)
                         for i, p in enumerate(params)]
        param_handles.append(Line2D([], [], color='tab:blue', lw=3, label='1-HMM'))
        param_handles.append(Line2D([], [], color='tab:red', lw=3, label='0-HMM'))
        ax.legend(handles=param_handles, fontsize=9.5, ncol=1, 
                  loc='center left', bbox_to_anchor=(1.02, 0.5),
                  frameon=True, edgecolor='0.8',
                  borderpad=0.4, handlelength=1.2, labelspacing=0.3)

    # ylabel centered per row
    fig.canvas.draw()
    for row_idx in range(2):
        row_axes = [ax for ax, _, r, c in all_axes if r == row_idx and c == 0]
        if not row_axes: continue
        ax = row_axes[0]
        pos = ax.get_position()
        fig.text(pos.x0 - 0.05, (pos.y0 + pos.y1) / 2,
                 'KL Divergence', fontsize=15,
                 va='center', ha='center', rotation=90)

    fig.suptitle(model_name, fontsize=16, y=0.98)
    plt.savefig(f'{PLOT_DIR}/kl_all_{model_key}.png', bbox_inches='tight')
    plt.show(); plt.close()